# ADAPT-ECG — Fase 2B: Entrenamiento mejorado (ResNet + SE + Focal Loss)

**Objetivo:** Entrenar desde cero un modelo mejorado sobre los 94,627 latidos MIT-BIH con:
- Ventana **128 samples** (355ms) en lugar de 71 — captura onda P para clase S
- Arquitectura **ResNet 1D + SE-Attention** — ~200K params, más expresiva que CNN simple
- **Focal Loss** — combate el desbalance severo (N=79%, F=0.8%)
- **WeightedRandomSampler** — batchs equilibrados durante el entrenamiento
- **OneCycleLR** — warmup + cosine decay, convergencia más rápida

**IMPORTANTE:** Ejecutar con GPU T4 (`Entorno de ejecución → Cambiar tipo → GPU`)  
Tiempo estimado: **20-30 minutos**

---
## CELDA 1 — Verificar GPU y instalar dependencias

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ADVERTENCIA: No hay GPU. Ve a Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> GPU')

---
## CELDA 2 — Montar Google Drive y cargar datos

**Antes de ejecutar esta celda**, sube los siguientes archivos a una carpeta `ADAPT-ECG/` en tu Google Drive:
- `data/processed/X_128.npy`
- `data/processed/y_128.npy`

In [ ]:
from google.colab import drive
import numpy as np

drive.mount('/content/drive')

# Ajusta esta ruta si pusiste los archivos en otra carpeta de Drive
DRIVE_DIR = '/content/drive/MyDrive/ADAPT-ECG/'

X = np.load(DRIVE_DIR + 'X_128.npy')
y = np.load(DRIVE_DIR + 'y_128.npy')

print(f'X: {X.shape}  dtype={X.dtype}')
print(f'y: {y.shape}  dtype={y.dtype}')
print(f'Memoria: {X.nbytes/1e6:.1f} MB')

---
## CELDA 3 — Distribución de clases y pesos para Focal Loss

Calculamos los pesos inversos de clase. Las clases raras (F, S) recibirán un peso mayor.

In [ ]:
import torch

CLASES = {0: 'N (Normal)', 1: 'S (Supraventricular)', 2: 'V (Ventricular)', 
          3: 'F (Fusion)', 4: 'Q (No clasif.)'}
N_CLASES = 5

print('Distribucion del dataset:')
print(f'  Total latidos: {len(y):,}')
conteos = []
for i in range(N_CLASES):
    c = int((y == i).sum())
    conteos.append(c)
    print(f'  Clase {i} {CLASES[i]}: {c:6,}  ({100*c/len(y):.1f}%)')

# Pesos para WeightedRandomSampler: inverso de frecuencia por muestra
conteos_arr = np.array(conteos, dtype=np.float32)
pesos_clase = 1.0 / conteos_arr
pesos_clase = pesos_clase / pesos_clase.sum() * N_CLASES  # normalizar
pesos_muestra = pesos_clase[y]  # peso de cada latido en el dataset

# Pesos alpha para Focal Loss (mismo criterio)
alpha = torch.tensor(pesos_clase, dtype=torch.float32).to(device)

print('\nPesos de clase (Focal Loss alpha):')
for i in range(N_CLASES):
    print(f'  Clase {i} ({CLASES[i][:1]}): {alpha[i].item():.4f}')

---
## CELDA 4 — Split train / val / test y DataLoaders

Split estratificado: **70% train / 15% val / 15% test**  
WeightedRandomSampler asegura que cada batch tenga representación de todas las clases.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split

BEAT_LEN   = 128
BATCH_SIZE = 128
SEED       = 42

# ---------------------------------------------------------------------------
# Dataset con augmentacion en entrenamiento
# ---------------------------------------------------------------------------
class ECGDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # (N, 1, 128)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            # Ruido gaussiano leve
            x += torch.randn_like(x) * 0.03
            # Escalado aleatorio de amplitud (0.85 - 1.15)
            x *= (0.85 + torch.rand(1).item() * 0.30)
        return x, self.y[idx]

# ---------------------------------------------------------------------------
# Split estratificado
# ---------------------------------------------------------------------------
idx = np.arange(len(y))

# Primero separa test (15%)
idx_trainval, idx_test = train_test_split(
    idx, test_size=0.15, stratify=y, random_state=SEED)

# Luego separa val del resto (15% del total ~ 17.6% de trainval)
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=0.176, stratify=y[idx_trainval], random_state=SEED)

print(f'Train: {len(idx_train):,}  Val: {len(idx_val):,}  Test: {len(idx_test):,}')

# Datasets
ds_train = ECGDataset(X[idx_train], y[idx_train], augment=True)
ds_val   = ECGDataset(X[idx_val],   y[idx_val],   augment=False)
ds_test  = ECGDataset(X[idx_test],  y[idx_test],  augment=False)

# WeightedRandomSampler para el loader de entrenamiento
pesos_train = torch.tensor(pesos_muestra[idx_train], dtype=torch.float32)
sampler = WeightedRandomSampler(pesos_train, num_samples=len(idx_train), replacement=True)

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=256,         shuffle=False,    num_workers=2, pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=256,         shuffle=False,    num_workers=2, pin_memory=True)

print(f'Batches de train por epoch: {len(dl_train)}')

---
## CELDA 5 — Arquitectura: ECG_ResNet_SE

```
Input (batch, 1, 128)
     Stem: Conv(1→32, k=7) + BN + ReLU          → (batch, 32, 128)
  ResBloque 1: ResBlock(32→32) + MaxPool(2)      → (batch, 32,  64)
  ResBloque 2: ResBlock(32→64) + MaxPool(2)      → (batch, 64,  32)
  ResBloque 3: ResBlock(64→128) + MaxPool(2)     → (batch, 128, 16)
  SE-Attention: recalibra canales                → (batch, 128, 16)
  AdaptiveAvgPool1d(1)                           → (batch, 128)
  Clasificador: Dense(128→64→5)
```

Total parametros: ~200K (vs 44K del modelo anterior)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# Bloque residual 1D
# ---------------------------------------------------------------------------
class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=5):
        super().__init__()
        pad = kernel // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel, padding=pad, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=pad, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        # Shortcut: proyeccion 1x1 si los canales cambian
        self.shortcut = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


# ---------------------------------------------------------------------------
# Squeeze-and-Excitation block
# ---------------------------------------------------------------------------
class SEBlock1D(nn.Module):
    def __init__(self, channels, ratio=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // ratio),
            nn.ReLU(),
            nn.Linear(channels // ratio, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).squeeze(-1)   # (batch, channels)
        e = self.fc(s).unsqueeze(-1)   # (batch, channels, 1)
        return x * e                   # recalibracion por canal


# ---------------------------------------------------------------------------
# Modelo principal
# ---------------------------------------------------------------------------
class ECG_ResNet_SE(nn.Module):
    def __init__(self, n_classes=5, input_len=128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )
        self.layer1 = nn.Sequential(ResBlock1D(32,  32),  nn.MaxPool1d(2))
        self.layer2 = nn.Sequential(ResBlock1D(32,  64),  nn.MaxPool1d(2))
        self.layer3 = nn.Sequential(ResBlock1D(64,  128), nn.MaxPool1d(2))
        self.se     = SEBlock1D(128, ratio=8)
        self.pool   = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.se(x)
        x = self.pool(x)
        return self.classifier(x)   # logits (sin softmax)


# Instanciar y mover a GPU
model = ECG_ResNet_SE(n_classes=5, input_len=128).to(device)

# Contar parametros
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametros entrenables: {total_params:,}')
print(model)

---
## CELDA 6 — Focal Loss + Optimizador + Scheduler

**Focal Loss**: `FL = -alpha * (1 - p_t)^gamma * log(p_t)`  
- `gamma=2`: enfoca el aprendizaje en ejemplos difíciles (bajo p_t)  
- `alpha`: pesos de clase calculados en Celda 3

In [ ]:
import torch.optim as optim

# ---------------------------------------------------------------------------
# Focal Loss
# ---------------------------------------------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha     = alpha      # tensor de pesos por clase
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # CE base: (N,)
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)          # probabilidad de la clase correcta
        focal = (1 - pt) ** self.gamma * ce

        if self.alpha is not None:
            at = self.alpha[targets]
            focal = at * focal

        return focal.mean() if self.reduction == 'mean' else focal.sum()


# ---------------------------------------------------------------------------
# Criterion, optimizador y scheduler
# ---------------------------------------------------------------------------
criterion = FocalLoss(alpha=alpha, gamma=2.0).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 50
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr        = 1e-3,
    steps_per_epoch = len(dl_train),
    epochs        = EPOCHS,
    pct_start     = 0.1,    # 10% warmup
    anneal_strategy = 'cos'
)

print('Criterion:', criterion)
print('Optimizer:', optimizer.__class__.__name__)
print(f'Scheduler: OneCycleLR  (max_lr=1e-3, epochs={EPOCHS}, warmup=10%)')

---
## CELDA 7 — Loop de entrenamiento

Incluye:
- Early stopping si `val_f1` no mejora en 8 épocas
- Guarda el mejor modelo por F1 macro (no por accuracy)
- Log por época: loss, accuracy, F1 macro

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import copy, time

# ---------------------------------------------------------------------------
# Funciones auxiliares
# ---------------------------------------------------------------------------
def evaluate(model, loader):
    model.eval()
    all_preds, all_true = [], []
    total_loss = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss   = criterion(logits, yb)
            total_loss += loss.item() * len(yb)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(yb.cpu().numpy())
    acc = accuracy_score(all_true, all_preds)
    f1  = f1_score(all_true, all_preds, average='macro', zero_division=0)
    return total_loss / len(loader.dataset), acc, f1


# ---------------------------------------------------------------------------
# Entrenamiento
# ---------------------------------------------------------------------------
PATIENCE    = 8
best_f1     = 0.0
best_state  = None
sin_mejora  = 0
historia    = []

print(f'{'Epoch':>5}  {'TrainLoss':>10}  {'ValLoss':>9}  {'ValAcc':>7}  {'ValF1':>7}  {'LR':>9}  {'Tiempo':>7}')
print('-' * 72)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_loss = 0.0

    for xb, yb in dl_train:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item() * len(yb)

    train_loss /= len(dl_train.dataset)
    val_loss, val_acc, val_f1 = evaluate(model, dl_val)
    lr_actual = scheduler.get_last_lr()[0]
    elapsed   = time.time() - t0

    historia.append({
        'epoch': epoch, 'train_loss': train_loss,
        'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1
    })

    marker = '  ← MEJOR' if val_f1 > best_f1 else ''
    print(f'{epoch:>5}  {train_loss:>10.4f}  {val_loss:>9.4f}  '
          f'{val_acc:>7.4f}  {val_f1:>7.4f}  {lr_actual:>9.2e}  '
          f'{elapsed:>5.1f}s{marker}')

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_state = copy.deepcopy(model.state_dict())
        sin_mejora = 0
    else:
        sin_mejora += 1
        if sin_mejora >= PATIENCE:
            print(f'\nEarly stopping en epoch {epoch} (sin mejora en {PATIENCE} epocas)')
            break

print(f'\nMejor Val F1 macro: {best_f1:.4f}')
model.load_state_dict(best_state)  # restaurar el mejor

---
## CELDA 8 — Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

epochs_log = [h['epoch']      for h in historia]
train_loss = [h['train_loss'] for h in historia]
val_loss   = [h['val_loss']   for h in historia]
val_acc    = [h['val_acc']    for h in historia]
val_f1     = [h['val_f1']     for h in historia]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_log, train_loss, label='Train')
axes[0].plot(epochs_log, val_loss,   label='Val')
axes[0].set_title('Focal Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_log, val_acc)
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim([0.85, 1.0])
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_log, val_f1, color='green')
axes[2].set_title('Validation F1 Macro')
axes[2].set_xlabel('Epoch')
axes[2].set_ylim([0.7, 1.0])
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/curvas_entrenamiento.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafica guardada en /content/curvas_entrenamiento.png')

---
## CELDA 9 — Evaluación completa en conjunto de test

Métricas finales por clase: precision, recall, F1.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in dl_test:
        xb = xb.to(device)
        preds = model(xb).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(yb.numpy())

nombres_clase = ['N', 'S', 'V', 'F', 'Q']
print('=' * 60)
print('RESULTADOS EN TEST SET')
print('=' * 60)
print(classification_report(all_true, all_preds,
                             target_names=nombres_clase,
                             digits=4))

# Comparacion con modelo anterior
modelo_anterior = {
    'accuracy': 0.9741,
    'f1_macro': 0.8954,
    'f1_N': 0.9858, 'f1_S': 0.7737, 'f1_V': 0.9470,
    'f1_F': 0.7932, 'f1_Q': 0.9774
}

from sklearn.metrics import f1_score, accuracy_score
nuevo_acc = accuracy_score(all_true, all_preds)
nuevo_f1  = f1_score(all_true, all_preds, average='macro', zero_division=0)
f1_por_clase = f1_score(all_true, all_preds, average=None, zero_division=0)

print('\n' + '=' * 60)
print('COMPARACION: Modelo anterior vs Nuevo')
print('=' * 60)
print(f'{'Metrica':<20} {'Anterior':>10} {'Nuevo':>10} {'Delta':>10}')
print('-' * 52)
print(f'{'Accuracy':<20} {modelo_anterior["accuracy"]:>10.4f} {nuevo_acc:>10.4f} {"+" if nuevo_acc > modelo_anterior["accuracy"] else ""}{nuevo_acc - modelo_anterior["accuracy"]:>+10.4f}')
print(f'{'F1 Macro':<20} {modelo_anterior["f1_macro"]:>10.4f} {nuevo_f1:>10.4f} {nuevo_f1 - modelo_anterior["f1_macro"]:>+10.4f}')
for i, cls in enumerate(nombres_clase):
    key = f'f1_{cls}'
    delta = f1_por_clase[i] - modelo_anterior[key]
    print(f'  F1 clase {cls:<12} {modelo_anterior[key]:>10.4f} {f1_por_clase[i]:>10.4f} {delta:>+10.4f}')

# Matriz de confusion
cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=nombres_clase, yticklabels=nombres_clase, ax=ax)
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusion — ECG ResNet-SE')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## CELDA 10 — Guardar el modelo + metadatos

Guarda el modelo en Drive y descarga una copia a tu máquina.

In [ ]:
import json
from datetime import datetime
from google.colab import files

# Nombre del archivo
MODEL_NAME = 'ecg_resnet_se_base.pth'
META_NAME  = 'ecg_resnet_se_meta.json'

# Guardar pesos
torch.save(model.state_dict(), f'/content/{MODEL_NAME}')
torch.save(model.state_dict(), DRIVE_DIR + MODEL_NAME)

# Metadatos para referencia futura
meta = {
    'nombre'       : 'ECG_ResNet_SE',
    'arquitectura' : 'ResNet1D + SE-Attention',
    'input_len'    : 128,
    'n_clases'     : 5,
    'clases'       : ['N', 'S', 'V', 'F', 'Q'],
    'params'       : total_params,
    'beat_before'  : 50,
    'beat_after'   : 78,
    'fs'           : 360,
    'loss'         : 'FocalLoss(gamma=2)',
    'optimizer'    : 'AdamW(lr=1e-3, wd=1e-4)',
    'scheduler'    : 'OneCycleLR',
    'epochs'       : len(historia),
    'best_val_f1'  : round(best_f1, 4),
    'test_accuracy': round(nuevo_acc, 4),
    'test_f1_macro': round(nuevo_f1, 4),
    'test_f1_por_clase': {c: round(float(f1_por_clase[i]), 4) for i, c in enumerate(nombres_clase)},
    'fecha'        : datetime.now().strftime('%Y-%m-%d %H:%M'),
    'dataset'      : 'MIT-BIH 48 registros, 94627 latidos'
}

with open(f'/content/{META_NAME}', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

with open(DRIVE_DIR + META_NAME, 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f'Modelo guardado: {MODEL_NAME}')
print(f'Metadatos guardados: {META_NAME}')
print(f'Copia en Drive: {DRIVE_DIR}')
print()
print('Descargando a tu maquina...')

# Descarga directa al PC
files.download(f'/content/{MODEL_NAME}')
files.download(f'/content/{META_NAME}')
files.download('/content/curvas_entrenamiento.png')
files.download('/content/confusion_matrix.png')

print('\nRESUMEN FINAL')
print(json.dumps(meta, indent=2, ensure_ascii=False))